# 03 · Filter & Rank — run the shared multi-layer filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 03** you filter the backbone pool on **foldability** (scRMSD/pLDDT) with
`design_type="monomer"`, and **report novelty (TM-score) as a coordinate, not a pass/fail**.

Run `00`–`02` first so `results/backbones.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. For
monomers the relevant cutoffs are `scrmsd` and `plddt`; `tm_to_pdb` is carried through and reported,
never used as a gate.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS['monomer']:", fp.DEFAULT_CUTOFFS["monomer"])
print("(scrmsd<2.0 foldable; plddt>85 confident; TM-score is REPORTED, not filtered)")

## Build `fp.Design` objects (monomer)

Map each backbone row onto an `fp.Design` with `design_type="monomer"`, populating `scrmsd`, `plddt`,
and `tm_to_pdb`. We pass `scrmsd` directly (computed in nb 02 from designed-vs-refolded Cα-RMSD), so
`self_consistency()` uses the provided value. We do **not** set `tm_to_pdb` as a cutoff anywhere —
it rides along for the frontier in nb 04.

In [ ]:
backbones = pd.read_csv("results/backbones.csv")

designs = []
for _, r in backbones.iterrows():
    designs.append(fp.Design(
        design_id=str(r["backbone_id"]),
        sequence="",                       # not needed for the foldability layers
        design_type="monomer",
        scrmsd=(None if pd.isna(r["scrmsd"]) else float(r["scrmsd"])),
        plddt=(None if pd.isna(r["plddt"]) else float(r["plddt"])),
        tm_to_pdb=(None if pd.isna(r["tm_to_pdb"]) else float(r["tm_to_pdb"])),
        extra={"length": int(r["length"]), "ss_bias": r["ss_bias"],
               "synthetic": bool(r.get("synthetic", False))},
    ))
print(len(designs), "Design objects built (design_type='monomer')")

## Run the pipeline (foldability gates; novelty reported)

`run_pipeline(..., design_type="monomer")` applies the layers in order and returns a ranked
DataFrame. We use the self-consistency layer (L1) as the foldability gate. (Layers 2–3 need an
orthogonal refold / solubility, added in nb 04; here L1 is the point.) `report()` prints hit-rate
accounting + the survival figure and writes the ranked CSV.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="monomer", use_layers=(1,))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj03")
print("\nNote: tm_to_pdb is shown for context — it is NOT a pass/fail criterion.")
top

## Survival + novelty-of-survivors (honest accounting)

How many backbones pass the foldability gate, and — crucially for this project — **what novelty did
the survivors have?** A filter that only passes conservative (high-TM) folds has spent no novelty
budget. Break the survival down by topology; that breakdown is the result.

In [ ]:
import rfdiff_tools as rt

passed = df_ranked[df_ranked["layers_passed"] >= 1]
print("foldable (passed L1):", len(passed), "/", len(df_ranked))
if "tm_to_pdb" in passed and passed["tm_to_pdb"].notna().any():
    novel_survivors = passed[passed["tm_to_pdb"] < rt.NOVEL_TM]
    print("  of which NOVEL (TM<0.5):", len(novel_survivors),
          "-> these are the novel-but-foldable designs the project is after")
# survival by topology (extra dict was flattened into the DataFrame by asdict)
if "extra" in df_ranked.columns:
    df_ranked["ss_bias"] = df_ranked["extra"].apply(
        lambda d: d.get("ss_bias") if isinstance(d, dict) else None)
    print("\nfoldable count by topology:")
    print(df_ranked.assign(folded=df_ranked["layers_passed"] >= 1)
          .groupby("ss_bias")["folded"].agg(["sum", "count"]))

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`filtering_pipeline`), not a one-off script.
- [ ] Foldability (scRMSD<2, pLDDT>85) used as the gate; **novelty (TM-score) reported, never filtered**.
- [ ] Survival reported, broken down by topology; novelty of the survivors noted.
- [ ] Any mapping assumptions (best-of-8 scRMSD, which refold tool) written down.

**Next:** `04_validate.ipynb` — the novelty-vs-scRMSD frontier + tool comparison.